# 📚 Fase 2 — Corpus, Chunking e Índice
## RAG para Normas Estruturais Brasileiras (NBR 6118 · 6120 · 6123)

> **Objetivo:** Transformar os PDFs normativos em unidades recuperáveis e rastreáveis, construir um índice vetorial denso (FAISS) e validar o retriever com Recall@k.

**Estrutura do notebook:**
- **Seção 0** — Instalação e configuração do ambiente
- **Seção 1** — Ingestão dos PDFs com metadados
- **Seção 2** — Chunking hierárquico e inspeção
- **Seção 3** — Geração de embeddings e índice FAISS
- **Seção 4** — Retrieval interativo (top-k configurável)
- **Seção 5** — Avaliação Recall@k com golden_set.json


---
## Seção 0 · Instalação e Configuração do Ambiente


In [6]:
# @title 0.1 · Instalar dependências
# Tempo estimado: ~3-5 min no Colab (download do modelo de embedding)
# Usa python -m pip para instalar no mesmo ambiente do kernel
!python -m pip install -q pdfplumber langchain-text-splitters sentence-transformers faiss-cpu tqdm pandas numpy
print('✅ Dependências instaladas.')

✅ Dependências instaladas.


In [4]:
# @title 0.2 · Montar Google Drive (necessário para acessar os PDFs)
import os
from pathlib import Path

# --- Montar o Drive ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('⚠️  Não está no Colab. Usando caminhos locais.')

# --- Configurar caminhos ---
# ⚙️ CONFIGURE AQUI: caminho para a pasta do projeto no seu Drive
if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/pln-rag-normas-estruturais')
else:
    # Execução local: sobe na árvore até achar raiz (dir com src/ e data/)
    cwd = Path('.').resolve()
    PROJECT_ROOT = cwd
    candidate = cwd
    for _ in range(5):  # evita loop infinito
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            PROJECT_ROOT = candidate
            break
        if candidate.parent == candidate:
            break
        candidate = candidate.parent

NORMS_DIR  = PROJECT_ROOT / 'data' / 'norms'
EVAL_DIR   = PROJECT_ROOT / 'data' / 'eval'
INDEX_DIR  = PROJECT_ROOT / 'index'
SRC_DIR    = PROJECT_ROOT / 'src'

# Garante que os diretórios existem
INDEX_DIR.mkdir(parents=True, exist_ok=True)
NORMS_DIR.mkdir(parents=True, exist_ok=True)

# Adiciona src/ ao path para importar os módulos
import sys
sys.path.insert(0, str(PROJECT_ROOT))

# Verifica PDFs disponíveis
pdfs = list(NORMS_DIR.glob('*.pdf'))
print(f'📂 Projeto: {PROJECT_ROOT}')
print(f'📄 PDFs encontrados: {len(pdfs)}')
for p in pdfs:
    print(f'   - {p.name} ({p.stat().st_size / 1024**2:.1f} MB)')

if len(pdfs) == 0:
    print('\n⚠️  Nenhum PDF encontrado em', NORMS_DIR)
    print('   Coloque os arquivos NBR6118_2023.pdf, NBR6120_2019.pdf e NBR6123_2023_PROPOSTA.pdf')
    print('   na pasta data/norms/ (no Drive no Colab ou localmente na raiz do projeto).')

⚠️  Não está no Colab. Usando caminhos locais.
📂 Projeto: C:\Users\IMTeixeira\Desktop\dev\pln-rag-normas-estruturais
📄 PDFs encontrados: 3
   - NBR6118_2023.pdf (3.1 MB)
   - NBR6120_2019.pdf (0.7 MB)
   - NBR6123_2023_PROPOSTA.pdf (9.0 MB)


---
## Seção 1 · Ingestão dos PDFs

Cada documento recebe metadados fixos definidos no módulo `ingestion.py`:

| doc_id | Título | Fonte | Edição |
|--------|--------|-------|--------|
| NBR6118 | Projeto de estruturas de concreto | ABNT | 2023 |
| NBR6120 | Ações para o cálculo de estruturas | ABNT | 2019 |
| NBR6123 | Forças devidas ao vento | ABNT | 2023 |


In [7]:
# @title 1.1 · Carregar todos os documentos
from src.ingestion import load_all_documents, print_document_summary

documents = load_all_documents(norms_dir=str(NORMS_DIR))
print(f'\n✅ {len(documents)} documentos carregados.')

Ingestão de normas:   0%|          | 0/3 [00:00<?, ?it/s]

[ingestion] Carregando NBR6118 (NBR6118_2023.pdf)...


Ingestão de normas:  33%|███▎      | 1/3 [00:18<00:36, 18.44s/it]

  ✓ NBR6118: 260 páginas, 3,424,809 caracteres
[ingestion] Carregando NBR6120 (NBR6120_2019.pdf)...


Ingestão de normas:  67%|██████▋   | 2/3 [00:23<00:10, 10.35s/it]

  ✓ NBR6120: 68 páginas, 172,337 caracteres
[ingestion] Carregando NBR6123 (NBR6123_2023_PROPOSTA.pdf)...


Ingestão de normas: 100%|██████████| 3/3 [00:24<00:00,  8.04s/it]

  ✓ NBR6123: 0 páginas, 0 caracteres

[ingestion] 3/3 documentos carregados.

✅ 3 documentos carregados.


In [8]:
# @title 1.2 · Inspecionar resumo de cada documento
for doc in documents:
    print_document_summary(doc)


  doc_id  : NBR6118
  titulo  : Projeto de estruturas de concreto — Procedimento
  fonte   : ABNT
  edicao  : 2023
  páginas : 260
  chars   : 3,424,809
  preview : NORMA ABNT NBR BRASILEIRA 6118 Quarta edição 28.08.2023 Projeto de estruturas de concreto Design of concrete structures ICS 91.080.40 ISBN 978-85-07-09632-0 Número de referência ABNT NBR 6118:2023 242 páginas © ABNT 2023  ABNT NBR 6118:2023 © ABNT 2023 (cid:55)(cid:82)(cid:71)(cid:82)(cid:86)(cid:3)(cid:82)(cid:86)(cid:3)(cid:71)(cid:76)(cid:85)(cid:72)(cid:76)(cid:87)(cid:82)(cid:86)(cid:3)(cid:85)(cid:72)(cid:86)(cid:72)(cid:85)(cid:89)(cid:68)(cid:71)(cid:82)(cid:86)(cid:17)(cid:3)(cid:36)(ci...

  doc_id  : NBR6120
  titulo  : Ações para o cálculo de estruturas de edificações
  fonte   : ABNT
  edicao  : 2019
  páginas : 68
  chars   : 172,337
  preview : lOMoARcPSD|9451152 66112200 22001199 CCoorrrriiggiiddaa EEssttrruuttuurraass DDee CCoonnccrreettoo II ((UUnniivveerrssiiddaaddee ddee MMooggii ddaass CCrruuzzeess)) S

In [9]:
# @title 1.3 · Visualizar amostra de texto extraído (com tabelas)
import pandas as pd

# Procura primeira página que contém uma tabela extraída
for doc in documents:
    for page in doc['pages']:
        if '[TABELA]' in page['text']:
            print(f"📊 Exemplo de tabela extraída | {doc['doc_id']} | Página {page['page_num']}")
            print('-' * 60)
            # Mostra trecho com a tabela
            start = page['text'].find('[TABELA]')
            print(page['text'][max(0, start-100):start+500])
            print('...')
            break
    else:
        continue
    break

📊 Exemplo de tabela extraída | NBR6118 | Página 35
------------------------------------------------------------
id:85)(cid:72)(cid:87)(cid:82)(cid:17)
© ABNT 2023 - Todos os direitos reservados (cid:20)(cid:26)

[TABELA]
(cid:38)(cid:79)(cid:68)(cid:86)(cid:86)(cid:72)(cid:3)(cid:71)(cid:72)(cid:3)
(cid:68)(cid:74)(cid:85)(cid:72)(cid:86)(cid:86)(cid:76)(cid:89)(cid:76)(cid:71)(cid:68)(cid:71)(cid:72)(cid:3)
(cid:68)(cid:80)(cid:69)(cid:76)(cid:72)(cid:81)(cid:87)(cid:68)(cid:79)	(cid:36)(cid:74)(cid:85)(cid:72)(cid:86)(cid:86)(cid:76)(cid:89)(cid:76)(cid:71)(cid:68)(cid:71)(cid:72)	(cid:38)(cid:79)(cid:68)(cid:86)(cid:86)(cid:76)(cid:191)(cid:70)(cid:68)(cid:111)(cid:109)(cid:82)(cid:3)(cid:74)(cid:72)
...


---
## Seção 2 · Chunking Hierárquico

**Parâmetros de segmentação (justificativa técnica):**
- `chunk_size = 800` chars → cobre uma tabela média de carga + parágrafo de contexto
- `chunk_overlap = 120` chars → ~1-2 linhas; preserva continuidade de enumerações e condições
- Separadores: `["\n\n", "\n", ". ", " "]` → prioriza quebras naturais de parágrafo

**Formato do `chunk_id`:** `{doc_id}#{secao_detectada}_{seq:04d}`  
Ex.: `NBR6120#3.2_0012`, `NBR6118#intro_0001`


In [10]:
# @title 2.1 · Gerar chunks de todos os documentos
from src.chunker import chunk_documents, print_chunks_stats, find_table_chunks

all_chunks = chunk_documents(documents)
print(f'\n✅ Total de chunks gerados: {len(all_chunks)}')

C:\Users\IMTeixeira\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
C:\Users\IMTeixeira\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[chunker] NBR6118: 5743 chunks (size=800, overlap=120)
[chunker] NBR6120: 303 chunks (size=800, overlap=120)
[chunker] NBR6123: 0 chunks (size=800, overlap=120)

[chunker] Total: 6046 chunks gerados.

✅ Total de chunks gerados: 6046


In [11]:
# @title 2.2 · Estatísticas de distribuição dos chunks
print_chunks_stats(all_chunks)

# Distribuição por tamanho (histograma texto)
import numpy as np
sizes = [c['n_chars'] for c in all_chunks]
bins = [0, 200, 400, 600, 800, 1000]
print('\n  Distribuição de tamanho dos chunks:')
counts, _ = np.histogram(sizes, bins=bins)
for i, (lo, hi) in enumerate(zip(bins[:-1], bins[1:])):
    bar = '█' * (counts[i] // max(1, max(counts) // 30))
    print(f'  {lo:4d}–{hi:4d} chars | {bar} {counts[i]}')


  ESTATÍSTICAS DOS CHUNKS
  Total de chunks : 6046
  Tamanho médio   : 605 chars
  Tamanho mediano : 696 chars
  Mínimo          : 1 chars
  Máximo          : 799 chars

  Por documento:
    NBR6118: 5743 chunks
    NBR6120: 303 chunks

  Distribuição de tamanho dos chunks:
     0– 200 chars | ███ 441
   200– 400 chars | ████ 604
   400– 600 chars | ████████ 1116
   600– 800 chars | ██████████████████████████████ 3885
   800–1000 chars |  0


In [12]:
# @title 2.3 · Inspecionar chunks com tabelas
table_chunks = find_table_chunks(all_chunks)
print(f'📊 Chunks que contêm tabelas: {len(table_chunks)}')

if table_chunks:
    # Mostra o primeiro chunk com tabela
    ex = table_chunks[0]
    print(f'\n  chunk_id : {ex["chunk_id"]}')
    print(f'  doc_id   : {ex["doc_id"]}')
    print(f'  seção    : {ex["secao"]}')
    print(f'  n_chars  : {ex["n_chars"]}')
    print(f'\n  Texto do chunk:')
    print('-' * 60)
    print(ex['texto'][:600])
    print('...')

📊 Chunks que contêm tabelas: 113

  chunk_id : NBR6118#intro_0722
  doc_id   : NBR6118
  seção    : intro
  n_chars  : 683

  Texto do chunk:
------------------------------------------------------------
[TABELA]
(cid:38)(cid:79)(cid:68)(cid:86)(cid:86)(cid:72)(cid:3)(cid:71)(cid:72)(cid:3)
(cid:68)(cid:74)(cid:85)(cid:72)(cid:86)(cid:86)(cid:76)(cid:89)(cid:76)(cid:71)(cid:68)(cid:71)(cid:72)(cid:3)
(cid:68)(cid:80)(cid:69)(cid:76)(cid:72)(cid:81)(cid:87)(cid:68)(cid:79)	(cid:36)(cid:74)(cid:85)(cid:72)(cid:86)(cid:86)(cid:76)(cid:89)(cid:76)(cid:71)(cid:68)(cid:71)(cid:72)	(cid:38)(cid:79)(cid:68)(cid:86)(cid:86)(cid:76)(cid:191)(cid:70)(cid:68)(cid:111)(cid:109)(cid:82)(cid:3)(cid:74)(cid:72)(cid:85)(cid:68)(cid:79)(cid:3)(cid:71)(cid:82)(cid:3)(cid:87)(cid:76)(cid:83)(cid:82)(cid:3)(cid:71
...


In [13]:
# @title 2.4 · Visualizar primeiros chunks por documento
from collections import defaultdict

by_doc = defaultdict(list)
for c in all_chunks:
    by_doc[c['doc_id']].append(c)

for doc_id, chunks in sorted(by_doc.items()):
    print(f'\n📘 {doc_id} — Primeiros 3 chunks:')
    for c in chunks[:3]:
        print(f'  [{c["chunk_id"]}] seção={c["secao"]} | {c["n_chars"]} chars')
        print(f'  {c["texto"][:120].strip()}...')
        print()


📘 NBR6118 — Primeiros 3 chunks:
  [NBR6118#intro_0001] seção=intro | 220 chars
  NORMA ABNT NBR
BRASILEIRA 6118
Quarta edição
28.08.2023
Projeto de estruturas de concreto
Design of concrete structures...

  [NBR6118#intro_0002] seção=intro | 30 chars
  ABNT NBR 6118:2023
© ABNT 2023...

  [NBR6118#intro_0003] seção=intro | 799 chars
  (cid:55)(cid:82)(cid:71)(cid:82)(cid:86)(cid:3)(cid:82)(cid:86)(cid:3)(cid:71)(cid:76)(cid:85)(cid:72)(cid:76)(cid:87)(c...


📘 NBR6120 — Primeiros 3 chunks:
  [NBR6120#intro_0001] seção=intro | 342 chars
  lOMoARcPSD|9451152
66112200 22001199 CCoorrrriiggiiddaa
EEssttrruuttuurraass DDee CCoonnccrreettoo II ((UUnniivveerrssii...

  [NBR6120#intro_0002] seção=intro | 591 chars
  Documento impresso em 17/01/2020 08:50:23, de uso elOMxoARcPScD|945115l2usivo de INSTITUTO FEDERAL DO ESPIRITO SANTO
NOR...

  [NBR6120#intro_0003] seção=intro | 748 chars
  Documento impresso em 17/01/2020 08:50:23, de uso elOMxoARcPScD|945115l2usivo de INSTITUTO FEDERAL DO ESPIR

---
## Seção 3 · Geração de Embeddings e Índice FAISS

**Modelo de embedding:** `neuralmind/bert-base-portuguese-cased`  
Treinado exclusivamente em português (BERT-PT-BR). Excelente para textos técnicos e normativos brasileiros sem necessidade de API externa.

**Índice FAISS:** `IndexFlatIP` com vetores L2-normalizados = busca por similaridade de cosseno exata.


In [14]:
# @title 3.1 · Carregar modelo de embedding
# ⏱️ Primeira execução: ~2-5 min para download do modelo (~400 MB)
from src.indexer import load_embedding_model, EMBEDDING_MODEL

model = load_embedding_model(EMBEDDING_MODEL)
print(f'\n✅ Modelo carregado: {EMBEDDING_MODEL}')
print(f'   Dimensão dos vetores: {model.get_sentence_embedding_dimension()}')

[indexer] Carregando modelo de embedding: neuralmind/bert-base-portuguese-cased


No sentence-transformers model found with name neuralmind/bert-base-portuguese-cased. Creating a new one with mean pooling.
C:\Users\IMTeixeira\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\IMTeixeira\.cache\huggingface\hub\models--neuralmind--bert-base-portuguese-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/en

[indexer] Modelo carregado. Dimensão dos vetores: 768

✅ Modelo carregado: neuralmind/bert-base-portuguese-cased
   Dimensão dos vetores: 768


In [ ]:
# @title 3.2 · Construir índice FAISS
# ⏱️ Estimativa: ~3-10 min dependendo do número de chunks e GPU
import time
from src.indexer import build_index, save_index

t0 = time.time()
faiss_index, indexed_chunks = build_index(all_chunks, model=model)
elapsed = time.time() - t0

print(f'\n✅ Índice construído em {elapsed:.1f}s')
print(f'   Vetores no índice: {faiss_index.ntotal}')

[indexer] Gerando embeddings para 6046 chunks...


Batches:   9%|▉         | 17/189 [05:28<53:34, 18.69s/it] 

In [ ]:
# @title 3.3 · Persistir índice e metadados em disco
save_index(faiss_index, indexed_chunks, index_dir=str(INDEX_DIR))
print('\n✅ Artefatos salvos:')
for f in INDEX_DIR.iterdir():
    print(f'   {f.name} ({f.stat().st_size / 1024:.1f} KB)')

---
## Seção 4 · Retrieval Interativo

Função `retrieve(query, k)` retorna os `k` chunks mais similares com:
- `rank` — posição no ranking
- `chunk_id` — ID rastreável (`NBRxxxx#secao_NNNN`)
- `score` — similaridade de cosseno [0, 1]
- `texto` — conteúdo do chunk


In [ ]:
# @title 4.1 · Testar retrieval com perguntas do golden set
from src.indexer import retrieve, print_retrieval_results

# Perguntas de teste representativas das categorias do golden set
test_queries = [
    ('factual_direta', 'Qual o valor típico de carga acidental para um pavimento de escritório?'),
    ('factual_direta', 'Como determinar a velocidade básica do vento em uma região?'),
    ('multi_trecho',   'Qual a diferença entre estados-limite últimos e estados-limite de serviço?'),
]

K_DEMO = 3  # @param {type: "slider", min: 1, max: 10, step: 1}

for categoria, query in test_queries:
    print(f'\n🔍 [{categoria}]')
    results = retrieve(query, faiss_index, indexed_chunks, model, k=K_DEMO)
    print_retrieval_results(results, query)

In [ ]:
# @title 4.2 · Comparar top-k = 3, 5 e 10 para a mesma query
from src.indexer import SUPPORTED_K

COMPARE_QUERY = 'Quando é permitido reduzir as cargas acidentais em um edifício?'  # @param {type: "string"}

print(f'Query: "{COMPARE_QUERY}"\n')
for k in SUPPORTED_K:
    results = retrieve(COMPARE_QUERY, faiss_index, indexed_chunks, model, k=k)
    doc_ids = [r['doc_id'] for r in results]
    chunk_ids = [r['chunk_id'] for r in results]
    scores = [f"{r['score']:.3f}" for r in results]
    print(f'  top-{k}: docs={doc_ids} | scores={scores}')
    print(f'         ids={chunk_ids}')

In [ ]:
# @title 4.3 · Query personalizada interativa
CUSTOM_QUERY = 'Como calcular o coeficiente de pressão do vento em uma edificação?'  # @param {type: "string"}
CUSTOM_K = 5  # @param {type: "slider", min: 1, max: 10, step: 1}

results = retrieve(CUSTOM_QUERY, faiss_index, indexed_chunks, model, k=CUSTOM_K)
print_retrieval_results(results, CUSTOM_QUERY)

---
## Seção 5 · Avaliação Recall@k

**Definição de Recall@k (baseline):**
```
hit(q, k) = 1  se doc_id de algum chunk em top-k corresponde à evidência esperada
Recall@k  = nº hits / nº perguntas avaliadas
```

**Nota metodológica:** A correspondência é feita por `doc_id` (conservadora). Após mapear os `chunk_id`s exatos para as seções do golden set, o recall real será maior.

Perguntas `fora_do_corpus` (evidência = null) são **excluídas** do cálculo.


In [ ]:
# @title 5.1 · Carregar golden set e verificar estrutura
from src.evaluator import load_golden_set
import pandas as pd

golden_set = load_golden_set(path=str(EVAL_DIR / 'golden_set.json'))

# Resumo por categoria
df_gs = pd.DataFrame(golden_set)
print('\nDistribuição por categoria:')
print(df_gs['categoria'].value_counts().to_string())
print(f'\nTotal: {len(df_gs)} perguntas')

In [ ]:
# @title 5.2 · Executar avaliação Recall@k
# ⏱️ Estimativa: ~30s (depende do número de perguntas e velocidade de GPU)
from src.evaluator import run_evaluation, print_evaluation_report, save_evaluation_report

# Cria função de retrieval com os artefatos já carregados
def retrieve_fn(query: str, k: int):
    return retrieve(query, faiss_index, indexed_chunks, model, k=k)

eval_results = run_evaluation(
    retrieve_fn=retrieve_fn,
    golden_set=golden_set,
    k_values=[3, 5, 10],
)

print('\n✅ Avaliação concluída.')

In [ ]:
# @title 5.3 · Exibir relatório completo
print_evaluation_report(eval_results)

In [ ]:
# @title 5.4 · Tabela detalhada por pergunta
detail_cols = ['id', 'categoria', 'evidencias', 'hit@3', 'hit@5', 'hit@10']
display(eval_results['details_df'][detail_cols])

In [ ]:
# @title 5.5 · Analisar perguntas onde retriever falhou (miss@10)
misses = eval_results['details_df'][~eval_results['details_df']['hit@10']]
print(f'❌ Perguntas com miss@10 (retriever não encontrou em top-10): {len(misses)}')
if len(misses) > 0:
    for _, row in misses.iterrows():
        print(f'\n  id={row["id"]} | [{row["categoria"]}]')
        print(f'  pergunta  : {row["pergunta"]}')
        print(f'  evidência : {row["evidencias"]}')
        docs_retrieved = row.get('retrieved_docs@10', 'N/A')
        print(f'  recuperou : {docs_retrieved}')

In [ ]:
# @title 5.6 · Salvar relatório de avaliação em disco
output_path = save_evaluation_report(eval_results, output_path=str(INDEX_DIR))
print(f'\n✅ Relatório salvo em: {output_path}')

# Exibe resumo final
print('\n📊 RECALL@K — RESUMO FINAL')
print('=' * 40)
for k, recall in eval_results['recall_at_k'].items():
    bar = '█' * int(recall * 20)
    print(f'  Recall@{k:2d} = {recall:.2%} |{bar}')
print('=' * 40)
print('\n✅ Fase 2 concluída com sucesso!')
print('   Próximo passo: Fase 3 — Pipeline RAG funcional')